In [1]:
# Paso 1: importar las librerias necesarias para explorar los datos
 
import pandas as pd
import numpy as np
from pathlib import Path
 
# Ignorar warnings
# ==============================================================================
import warnings
warnings.filterwarnings("ignore")


# Configuración
# -----------------------------------------------------------------------
pd.set_option('display.max_columns', None) # para poder visualizar todas las columnas de los DataFrames

In [2]:
# Paso 2: definir las rutas de entrada y salida de los datos

RUTA_RAW = Path("../data/raw")
RUTA_INTERIM = Path("../data/interim")

RUTA_INTERIM.mkdir(parents=True, exist_ok=True)

In [3]:
from pathlib import Path
import chardet


def detectar_encodings_archivos(lista_archivos):
    """
    Detecta el encoding de varios archivos.

    Parametros:
        lista_archivos: lista de rutas de archivos.
    """

    for archivo in lista_archivos:

        with open(archivo, "rb") as f:
            resultado = chardet.detect(f.read())

        print("-" * 50)
        print(f"Archivo: {Path(archivo).name}")
        print(f"Encoding: {resultado['encoding']}")
        print(f"Confianza: {resultado['confidence']:.2%}")

In [4]:
archivos = [
    "../data/raw/brazil_public_holidays_comas.csv",
    "../data/raw/olist_customers_dataset.csv",
    "../data/raw/olist_orders_dataset.csv",
    "../data/raw/olist_order_items_dataset.csv",
    "../data/raw/olist_order_payments_dataset.csv",
    "../data/raw/olist_products_dataset.csv",
    "../data/raw/product_category_name_translation.csv"
]

detectar_encodings_archivos(archivos)

--------------------------------------------------
Archivo: brazil_public_holidays_comas.csv
Encoding: utf-8
Confianza: 86.38%
--------------------------------------------------
Archivo: olist_customers_dataset.csv
Encoding: ascii
Confianza: 100.00%
--------------------------------------------------
Archivo: olist_orders_dataset.csv
Encoding: ascii
Confianza: 100.00%
--------------------------------------------------
Archivo: olist_order_items_dataset.csv
Encoding: ascii
Confianza: 100.00%
--------------------------------------------------
Archivo: olist_order_payments_dataset.csv
Encoding: ascii
Confianza: 100.00%
--------------------------------------------------
Archivo: olist_products_dataset.csv
Encoding: ascii
Confianza: 100.00%
--------------------------------------------------
Archivo: product_category_name_translation.csv
Encoding: UTF-8-SIG
Confianza: 100.00%


In [5]:
# Paso 3: cargar los datasets originales

def cargar_csv(ruta, **kwargs):
    """
    Carga un archivo CSV.

    Parametros:
        ruta: ruta del archivo.
        **kwargs: parametros adicionales para pd.read_csv().

    Devuelve:
        DataFrame.
    """

    return pd.read_csv(ruta, **kwargs)


customers = cargar_csv(RUTA_RAW / "olist_customers_dataset.csv")
orders = cargar_csv(RUTA_RAW / "olist_orders_dataset.csv")
items = cargar_csv(RUTA_RAW / "olist_order_items_dataset.csv")
payments = cargar_csv(RUTA_RAW / "olist_order_payments_dataset.csv")
products = cargar_csv(RUTA_RAW / "olist_products_dataset.csv")
categories = cargar_csv(RUTA_RAW / "product_category_name_translation.csv",  encoding="utf-8")
holidays = cargar_csv(RUTA_RAW / "brazil_public_holidays_comas.csv",  encoding="utf-8")

In [6]:
# Paso 4: crear copias para no modificar los datasets originales

customers_clean = customers.copy()
orders_clean = orders.copy()
items_clean = items.copy()
payments_clean = payments.copy()
products_clean = products.copy()
categories_clean = categories.copy()
holidays_clean = holidays.copy()

In [7]:
# Paso 5: revisar los tipos de datos antes de transformar

def resumen_tipos(df, nombre_dataset):
    """
    Muestra los tipos de datos de un DataFrame.

    Parametros:
        df: DataFrame a analizar.
        nombre_dataset: nombre identificativo del dataset.

    Devuelve:
        DataFrame con columna y tipo de dato.
    """
    return pd.DataFrame({
        "dataset": nombre_dataset,
        "columna": df.columns,
        "tipo_dato": df.dtypes.astype(str).values
    })


tipos_iniciales = pd.concat([
    resumen_tipos(customers_clean, "customers"),
    resumen_tipos(orders_clean, "orders"),
    resumen_tipos(items_clean, "items"),
    resumen_tipos(payments_clean, "payments"),
    resumen_tipos(products_clean, "products"),
    resumen_tipos(categories_clean, "categories"),
    resumen_tipos(holidays_clean, "holidays")
])

tipos_iniciales

,dataset,columna,tipo_dato
0,customers,customer_id,object
1,customers,customer_unique_id,object
2,customers,customer_zip_code_prefix,int64
3,customers,customer_city,object
4,customers,customer_state,object
0,orders,order_id,object
1,orders,customer_id,object
2,orders,order_status,object
3,orders,order_purchase_timestamp,object
4,orders,order_approved_at,object


In [8]:
# Paso 6: revisar los valores nulos antes de aplicar tratamientos

def resumen_nulos(df, nombre_dataset):
    """
    Calcula el numero y porcentaje de valores nulos por columna.

    Parametros:
        df: DataFrame a analizar.
        nombre_dataset: nombre del dataset.

    Devuelve:
        DataFrame con el resumen de nulos.
    """
    resumen = pd.DataFrame({
        "dataset": nombre_dataset,
        "columna": df.columns,
        "nulos": df.isna().sum().values,
        "porcentaje_nulos": (df.isna().mean().values * 100).round(2)
    })

    return resumen.sort_values(by="porcentaje_nulos", ascending=False)


nulos_iniciales = pd.concat([
    resumen_nulos(customers_clean, "customers"),
    resumen_nulos(orders_clean, "orders"),
    resumen_nulos(items_clean, "items"),
    resumen_nulos(payments_clean, "payments"),
    resumen_nulos(products_clean, "products"),
    resumen_nulos(categories_clean, "categories"),
    resumen_nulos(holidays_clean, "holidays")
])

nulos_iniciales[nulos_iniciales["nulos"] > 0]

,dataset,columna,nulos,porcentaje_nulos
6,orders,order_delivered_customer_date,2965,2.98
5,orders,order_delivered_carrier_date,1783,1.79
4,orders,order_approved_at,160,0.16
1,products,product_category_name,610,1.85
3,products,product_description_lenght,610,1.85
2,products,product_name_lenght,610,1.85
4,products,product_photos_qty,610,1.85
5,products,product_weight_g,2,0.01
7,products,product_height_cm,2,0.01
6,products,product_length_cm,2,0.01


In [9]:
# Paso 7: eliminar columnas que no aportan informacion analitica

def eliminar_columnas(df, columnas):
    """
    Elimina columnas de un DataFrame si existen.

    Parametros:
        df: DataFrame de entrada.
        columnas: lista de columnas a eliminar.

    Devuelve:
        DataFrame sin las columnas indicadas.
    """
    columnas_existentes = [col for col in columnas if col in df.columns]

    return df.drop(columns=columnas_existentes)


holidays_clean = eliminar_columnas(
    holidays_clean,
    ["Unnamed: 0", "isPaidTimeOff"]
)

holidays_clean.head()


,countryOrRegion,holidayName,normalizeHolidayName,countryRegionCode,date
0,Brazil,Carnaval,Carnaval,BR,10/02/1970
1,Brazil,Quarta-feira de cinzas (InÃ­cio da Quaresma),Quarta-feira de cinzas (InÃ­cio da Quaresma),BR,11/02/1970
2,Brazil,Sexta-feira Santa,Sexta-feira Santa,BR,27/03/1970
3,Brazil,PÃ¡scoa,PÃ¡scoa,BR,29/03/1970
4,Brazil,Tiradentes,Tiradentes,BR,21/04/1970


In [10]:
# Paso 8: corregir problemas de encoding en los datasets
 
def corregir_encoding(texto):
    """
    Corrige problemas de codificación en cadenas de texto que han sido
    interpretadas incorrectamente entre los formatos Latin-1 y UTF-8.

    Esta función es útil para recuperar caracteres especiales que aparecen
    corruptos tras la lectura de archivos CSV, por ejemplo:

    - "Pã¡scoa" -> "Páscoa"
    - "Independãªncia" -> "Independência"
    - "Proclamaã§ã£o" -> "Proclamação"

    Parámetros
    ----------
    texto : str
        Cadena de texto a corregir.

    Devuelve
    --------
    str
        Texto corregido. Si la conversión no es posible o el valor es nulo,
        devuelve el valor original.
    """

    if pd.isna(texto):
        return texto

    try:
        return texto.encode("latin1").decode("utf-8")
    except Exception:
        return texto

In [11]:
# Paso 9: aplicar la función de corrección de encoding a las columnas relevantes
 
holidays_clean["holidayName"] = holidays_clean["holidayName"].apply(corregir_encoding)

holidays_clean["normalizeHolidayName"] = (
    holidays_clean["normalizeHolidayName"]
    .apply(corregir_encoding)
)

In [12]:
# Paso 10: revisar los nombres de los feriados y sus codificaciones
 
holidays_clean[
    ["holidayName"]
].drop_duplicates().head(20)

,holidayName
0,Carnaval
1,Quarta-feira de cinzas (Início da Quaresma)
2,Sexta-feira Santa
3,Páscoa
4,Tiradentes
5,Dia Mundial do Trabalho
6,Corpus Christi
7,Independência do Brasil
8,Nossa Senhora Aparecida
9,Finados


In [13]:
# Paso 11: normalizar columnas de texto eliminando espacios y pasando a minusculas

def limpiar_texto(df, columnas):
    """
    Limpia columnas de texto eliminando espacios al inicio y final
    y convirtiendo los valores a minusculas.

    Parametros:
        df: DataFrame de entrada.
        columnas: lista de columnas de texto.

    Devuelve:
        DataFrame con columnas de texto normalizadas.
    """
    df = df.copy()

    for columna in columnas:
        if columna in df.columns:
            df[columna] = (
                df[columna]
                .astype("string")
                .str.strip()
                .str.lower()
            )

    return df


customers_clean = limpiar_texto(
    customers_clean,
    ["customer_city", "customer_state"]
)

orders_clean = limpiar_texto(
    orders_clean,
    ["order_status"]
)

products_clean = limpiar_texto(
    products_clean,
    ["product_category_name"]
)

categories_clean = limpiar_texto(
    categories_clean,
    ["product_category_name", "product_category_name_english"]
)

holidays_clean = limpiar_texto(
    holidays_clean,
    [
        "countryOrRegion",
        "holidayName",
        "normalizeHolidayName",
        "countryRegionCode"
    ]
)

In [14]:
# Paso 12: convertir identificadores a texto porque son claves y no variables numericas

def convertir_a_texto(df, columnas):
    """
    Convierte columnas a tipo object.

    Parametros:
        df: DataFrame de entrada.
        columnas: lista de columnas a convertir.

    Devuelve:
        DataFrame con las columnas convertidas a object.
    """
    df = df.copy()

    for columna in columnas:
        if columna in df.columns:
            df[columna] = df[columna].astype("object")

    return df


customers_clean = convertir_a_texto(
    customers_clean,
    [
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state"
    ]
)

orders_clean = convertir_a_texto(
    orders_clean,
    [
        "order_id",
        "customer_id",
        "order_status"
    ]
)

items_clean = convertir_a_texto(
    items_clean,
    [
        "order_id",
        "product_id",
        "seller_id"
    ]
)

payments_clean = convertir_a_texto(
    payments_clean,
    [
        "order_id",
        "payment_type"
    ]
)

products_clean = convertir_a_texto(
    products_clean,
    [
        "product_id",
        "product_category_name"
    ]
)

categories_clean = convertir_a_texto(
    categories_clean,
    [
        "product_category_name",
        "product_category_name_english"
    ]
)

holidays_clean = convertir_a_texto(
    holidays_clean,
    [
        "countryOrRegion",
        "holidayName",
        "normalizeHolidayName",
        "countryRegionCode"
    ]
)

In [15]:
# Paso 13: convertir columnas de fecha a formato datetime

def convertir_fechas(df, columnas):
    """
    Convierte columnas a formato datetime.

    Parametros:
        df: DataFrame de entrada.
        columnas: lista de columnas de fecha.

    Devuelve:
        DataFrame con columnas convertidas a datetime.
    """
    df = df.copy()

    for columna in columnas:
        if columna in df.columns:
            df[columna] = pd.to_datetime(
                df[columna],
                errors="coerce"
            )

    return df


orders_clean = convertir_fechas(
    orders_clean,
    [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
)

items_clean = convertir_fechas(
    items_clean,
    ["shipping_limit_date"]
)

# Conversión específica para la fecha de festivos

holidays_clean["date"] = pd.to_datetime(
    holidays_clean["date"],
    errors="coerce",
    dayfirst=True
)

In [16]:
# Paso 14: comprobar que las columnas de fecha se han convertido correctamente

orders_clean[
    [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
].dtypes

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

In [17]:
# Paso 15.1: revisar rango temporal de pedidos

print("Fecha minima pedido:", orders_clean["order_purchase_timestamp"].min())
print("Fecha maxima pedido:", orders_clean["order_purchase_timestamp"].max())

Fecha minima pedido: 2016-09-04 21:15:19
Fecha maxima pedido: 2018-10-17 17:30:18


In [18]:
# Paso 15.2: revisar rango temporal de festivos

print("Fecha minima festivo:", holidays_clean["date"].min())
print("Fecha maxima festivo:", holidays_clean["date"].max())

Fecha minima festivo: 1970-02-10 00:00:00
Fecha maxima festivo: 2098-12-25 00:00:00


In [19]:
# Paso 16: filtrar los festivos al periodo temporal de los pedidos

anios_pedidos = orders_clean["order_purchase_timestamp"].dt.year.dropna().unique()

holidays_clean = holidays_clean[
    holidays_clean["date"].dt.year.isin(anios_pedidos)
].copy()

print("Anios de pedidos:")
print(sorted(anios_pedidos))

print("Anios de festivos tras el filtro:")
print(sorted(holidays_clean["date"].dt.year.unique()))

print("Numero de festivos tras el filtro:", holidays_clean.shape[0])

Anios de pedidos:
[np.int32(2016), np.int32(2017), np.int32(2018)]
Anios de festivos tras el filtro:
[np.int32(2016), np.int32(2017), np.int32(2018)]
Numero de festivos tras el filtro: 39


In [20]:
# Paso 17: renombrar la columna date para facilitar la futura union

holidays_clean = holidays_clean.rename(
    columns={"date": "holiday_date"}
)

holidays_clean.head()

,countryOrRegion,holidayName,normalizeHolidayName,countryRegionCode,holiday_date
596,brazil,ano novo,ano novo,br,2016-01-01
597,brazil,carnaval,carnaval,br,2016-02-09
598,brazil,quarta-feira de cinzas (início da quaresma),quarta-feira de cinzas (início da quaresma),br,2016-02-10
599,brazil,sexta-feira santa,sexta-feira santa,br,2016-03-25
600,brazil,páscoa,páscoa,br,2016-03-27


In [21]:
# Paso 17.1: comprobar el rango final de festivos

print("Fecha minima festivo:", holidays_clean["holiday_date"].min())
print("Fecha maxima festivo:", holidays_clean["holiday_date"].max())

Fecha minima festivo: 2016-01-01 00:00:00
Fecha maxima festivo: 2018-12-25 00:00:00


In [22]:
# Paso 18: tratar productos sin categoria asignando una categoria generica

products_clean["product_category_name"] = (
    products_clean["product_category_name"]
    .fillna("sin_categoria")
)

products_clean["product_category_name"].isna().sum()

np.int64(0)

In [23]:
# Paso 19: imputar con 0 las columnas descriptivas sin informacion

columnas_descriptivas = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty"
]

products_clean[columnas_descriptivas] = (
    products_clean[columnas_descriptivas].fillna(0)
)

products_clean[columnas_descriptivas].isna().sum()

product_name_lenght           0
product_description_lenght    0
product_photos_qty            0
dtype: int64

In [24]:
# Paso 20: eliminar productos con nulos en variables fisicas porque solo afectan a 2 registros

columnas_fisicas = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for columna in columnas_fisicas:

    products_clean[columna] = (
        products_clean[columna]
        .fillna(products_clean[columna].median())
    )

In [25]:
# Paso 21: convertir columnas numericas de producto a entero tras tratar los nulos

columnas_enteras_products = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

products_clean[columnas_enteras_products] = (
    products_clean[columnas_enteras_products].astype("int64")
)

products_clean[columnas_enteras_products].dtypes

product_name_lenght           int64
product_description_lenght    int64
product_photos_qty            int64
product_weight_g              int64
product_length_cm             int64
product_height_cm             int64
product_width_cm              int64
dtype: object

In [26]:
# Paso 22: identificar categorias de productos sin traduccion al ingles

categorias_products = set(
    products_clean["product_category_name"].dropna().unique()
)

categorias_translation = set(
    categories_clean["product_category_name"].dropna().unique()
)

categorias_sin_traduccion = categorias_products - categorias_translation

categorias_sin_traduccion

{'pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos', 'sin_categoria'}

In [27]:
# Paso 23: añadir traduccion manual para la categoria creada durante la limpieza

nueva_categoria = pd.DataFrame({
    "product_category_name": ["sin_categoria"],
    "product_category_name_english": ["without_category"]
})

categories_clean = pd.concat(
    [categories_clean, nueva_categoria],
    ignore_index=True
)

categories_clean.tail()

,product_category_name,product_category_name_english
67,artes_e_artesanato,arts_and_craftmanship
68,fraldas_higiene,diapers_and_hygiene
69,fashion_roupa_infanto_juvenil,fashion_childrens_clothes
70,seguros_e_servicos,security_and_services
71,sin_categoria,without_category


In [28]:
# Paso 24: comprobar valores negativos o incoherentes en precios y costes de envio

print("Precios negativos:", (items_clean["price"] < 0).sum())
print("Costes de envio negativos:", (items_clean["freight_value"] < 0).sum())
print("Precios iguales a cero:", (items_clean["price"] == 0).sum())
print("Costes de envio iguales a cero:", (items_clean["freight_value"] == 0).sum())

Precios negativos: 0
Costes de envio negativos: 0
Precios iguales a cero: 0
Costes de envio iguales a cero: 383


In [29]:
# Paso 24.1: analizar los registros con coste de envio igual a cero

items_clean.loc[
    items_clean["freight_value"] == 0,
    ["order_id", "product_id", "price", "freight_value"]
].head()

,order_id,product_id,price,freight_value
114,00404fa7a687c8c44ca69d42695aae73,53b36df67ebb7c41585e8d54d6772e08,99.9,0.0
258,00a870c6c06346e85335524935c600c0,aca2eb7d00ea1a7b8ebd4e68314663af,69.9,0.0
483,011c899816ea29773525bd3322dbb6aa,53b36df67ebb7c41585e8d54d6772e08,99.9,0.0
508,012b3f6ab7776a8ab3443a4ad7bef2e6,422879e10f46682990de24d770e7f83d,53.9,0.0
509,012b3f6ab7776a8ab3443a4ad7bef2e6,422879e10f46682990de24d770e7f83d,53.9,0.0


In [30]:
# Paso 24.2: porcentaje de registros con envio gratuito

envios_gratis = (
    items_clean["freight_value"] == 0
).sum()

porcentaje = (
    envios_gratis / len(items_clean)
) * 100

print(f"Registros con envio gratuito: {envios_gratis}")
print(f"Porcentaje: {porcentaje:.2f}%")

Registros con envio gratuito: 383
Porcentaje: 0.34%


In [31]:
# Paso 24.3: productos con mayor frecuencia de envio gratuito

(
    items_clean.loc[
        items_clean["freight_value"] == 0
    ]
    ["product_id"]
    .value_counts()
    .head(10)
)

product_id
53b36df67ebb7c41585e8d54d6772e08    187
aca2eb7d00ea1a7b8ebd4e68314663af     98
422879e10f46682990de24d770e7f83d     56
7a10781637204d8d10485c71a6108a2e     27
f1c7f353075ce59d8a6f3cf58f419c9c      9
5a848e4ab52fd5445cdc07aab1c40e48      2
81fe540cb0119e1d4ef5f191701b3cb9      1
2b4609f8948be18874494203496bc318      1
2a34e0af5f72ca6cdeb148377a247c86      1
4fcb3d9a5f4871e8362dfedbdb02b064      1
Name: count, dtype: int64

In [32]:
# Paso 24.4: estadisticos de precio para envios gratuitos

items_clean.loc[
    items_clean["freight_value"] == 0,
    "price"
].describe()

count    383.000000
mean      98.601488
std       50.004247
min       53.900000
25%       69.900000
50%       99.900000
75%      106.900000
max      712.900000
Name: price, dtype: float64

In [33]:
# Paso 25: comprobar valores negativos o incoherentes en pagos

print("Pagos negativos:", (payments_clean["payment_value"] < 0).sum())
print("Pagos iguales a cero:", (payments_clean["payment_value"] == 0).sum())
print("Cuotas negativas:", (payments_clean["payment_installments"] < 0).sum())
print("Cuotas iguales a cero:", (payments_clean["payment_installments"] == 0).sum())

Pagos negativos: 0
Pagos iguales a cero: 9
Cuotas negativas: 0
Cuotas iguales a cero: 2


In [34]:
# Paso 25.1: analizar pagos con importe igual a cero

payments_clean.loc[
    payments_clean["payment_value"] == 0
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0


In [35]:
# Paso 25.2: analizar cuotas iguales a cero

payments_clean.loc[
    payments_clean["payment_installments"] == 0
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [36]:
# Paso 25.3: comprobar si los registros coinciden

payments_clean.loc[
    (payments_clean["payment_value"] == 0)
    |
    (payments_clean["payment_installments"] == 0)
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.00
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.00
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.00
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.00
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.00
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.00
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.00
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.00


In [37]:
# Analizar registros con cuotas iguales a cero

payments_clean.loc[
    payments_clean["payment_installments"] == 0
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [38]:
porcentaje = (
    (payments_clean["payment_installments"] == 0).sum()
    / len(payments_clean)
) * 100

print(f"{porcentaje:.5f}%")

0.00193%


In [39]:
# Corregir cuotas iguales a cero

payments_clean["payment_installments"] = (
    payments_clean["payment_installments"]
    .replace(0, 1)
)

In [40]:
# Analizar registros con cuotas iguales a cero

payments_clean.loc[
    payments_clean["payment_installments"] == 0
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value


In [41]:
# Paso 26: asegurar que variables enteras relevantes mantienen el tipo adecuado

items_clean["order_item_id"] = items_clean["order_item_id"].astype("int64")

payments_clean["payment_sequential"] = (
    payments_clean["payment_sequential"].astype("int64")
)

payments_clean["payment_installments"] = (
    payments_clean["payment_installments"].astype("int64")
)

In [42]:
# Paso 27: revisar nulos finales despues del tratamiento aplicado

datasets_clean = {
    "customers": customers_clean,
    "orders": orders_clean,
    "items": items_clean,
    "payments": payments_clean,
    "products": products_clean,
    "categories": categories_clean,
    "holidays": holidays_clean
}

nulos_finales = pd.concat([
    resumen_nulos(df, nombre)
    for nombre, df in datasets_clean.items()
])

nulos_finales[nulos_finales["nulos"] > 0]

,dataset,columna,nulos,porcentaje_nulos
6,orders,order_delivered_customer_date,2965,2.98
5,orders,order_delivered_carrier_date,1783,1.79
4,orders,order_approved_at,160,0.16


Los valores nulos restantes se concentran únicamente en columnas de fecha del dataset orders. Estos nulos están relacionados con pedidos no completados, cancelados, no disponibles o todavía en proceso. Por tanto, se consideran nulos estructurales derivados del ciclo de vida del pedido y se mantienen sin imputar para no introducir fechas artificiales.

In [43]:
# Paso 27.1: analizar los nulos de fechas segun el estado del pedido

columnas_fecha_con_nulos = [
    "order_delivered_customer_date",
    "order_delivered_carrier_date",
    "order_approved_at"
]

for columna in columnas_fecha_con_nulos:
    print(f"\nEstados de pedido con nulos en {columna}")
    display(
        orders_clean.loc[
            orders_clean[columna].isna(),
            "order_status"
        ].value_counts()
    )


Estados de pedido con nulos en order_delivered_customer_date


order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64


Estados de pedido con nulos en order_delivered_carrier_date


order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64


Estados de pedido con nulos en order_approved_at


order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

In [44]:
# Paso 28: revisar tipos finales despues de las conversiones aplicadas

tipos_finales = pd.concat([
    resumen_tipos(df, nombre)
    for nombre, df in datasets_clean.items()
])

tipos_finales

,dataset,columna,tipo_dato
0,customers,customer_id,object
1,customers,customer_unique_id,object
2,customers,customer_zip_code_prefix,object
3,customers,customer_city,object
4,customers,customer_state,object
0,orders,order_id,object
1,orders,customer_id,object
2,orders,order_status,object
3,orders,order_purchase_timestamp,datetime64[ns]
4,orders,order_approved_at,datetime64[ns]


In [45]:
# Paso 28.1: comprobar si las columnas datetime contienen componente horaria

columnas_fecha = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for columna in columnas_fecha:

    horas_distintas = (
        orders_clean[columna]
        .dropna()
        .dt.hour
        .nunique()
    )

    print(
        f"{columna}: {horas_distintas} horas distintas"
    )

order_purchase_timestamp: 24 horas distintas
order_approved_at: 24 horas distintas
order_delivered_carrier_date: 24 horas distintas
order_delivered_customer_date: 24 horas distintas
order_estimated_delivery_date: 1 horas distintas


In [46]:
# Paso 29: revisar si existen duplicados completos despues de la limpieza

def resumen_duplicados(datasets):
    """
    Calcula el numero de filas duplicadas por dataset.

    Parametros:
        datasets: diccionario de DataFrames.

    Devuelve:
        DataFrame con filas totales y duplicados.
    """
    resumen = []

    for nombre, df in datasets.items():
        resumen.append({
            "dataset": nombre,
            "filas": len(df),
            "duplicados": df.duplicated().sum()
        })

    return pd.DataFrame(resumen)


resumen_duplicados(datasets_clean)

,dataset,filas,duplicados
0,customers,99441,0
1,orders,99441,0
2,items,112650,0
3,payments,103886,0
4,products,32951,0
5,categories,72,0
6,holidays,39,0


In [47]:
# Paso 29.1: validar clave primaria customers

customers_clean["customer_id"].duplicated().sum()

np.int64(0)

In [48]:
# Paso 29.2: validar clave primaria orders

orders_clean["order_id"].duplicated().sum()

np.int64(0)

In [49]:
# Paso 29.3: validar clave primaria products

products_clean["product_id"].duplicated().sum()

np.int64(0)

In [50]:
# Paso 29.4: validar clave primaria categories

categories_clean["product_category_name"].duplicated().sum()

np.int64(0)

In [51]:
# Paso 29.5: pedidos sin cliente asociado

clientes_huerfanos = (
    set(orders_clean["customer_id"])
    - set(customers_clean["customer_id"])
)

len(clientes_huerfanos)

0

In [52]:
# Paso 29.6: lineas de pedido sin pedido asociado

items_huerfanos = (
    set(items_clean["order_id"])
    - set(orders_clean["order_id"])
)

len(items_huerfanos)

0

In [53]:
# Paso 29.7: pagos sin pedido asociado

payments_huerfanos = (
    set(payments_clean["order_id"])
    - set(orders_clean["order_id"])
)

len(payments_huerfanos)

0

In [54]:
# Paso 29.8: productos vendidos sin registro en products

products_huerfanos = (
    set(items_clean["product_id"])
    - set(products_clean["product_id"])
)

len(products_huerfanos)

0

In [55]:
# Paso 29.9: resumen de integridad referencial

integridad = pd.DataFrame({
    "relacion": [
        "orders-customers",
        "items-orders",
        "payments-orders",
        "items-products"
    ],
    "registros_huerfanos": [
        len(clientes_huerfanos),
        len(items_huerfanos),
        len(payments_huerfanos),
        len(products_huerfanos)
    ]
})

integridad

,relacion,registros_huerfanos
0,orders-customers,0
1,items-orders,0
2,payments-orders,0
3,items-products,0


In [56]:
# Paso 30: guardar los datasets limpios en la carpeta data/interim en formato pickle

customers_clean.to_pickle(
    RUTA_INTERIM / "customers_clean.pkl"
)

orders_clean.to_pickle(
    RUTA_INTERIM / "orders_clean.pkl"
)


items_clean.to_pickle(
    RUTA_INTERIM / "items_clean.pkl"
)

payments_clean.to_pickle(
    RUTA_INTERIM / "payments_clean.pkl"
)

products_clean.to_pickle(
    RUTA_INTERIM / "products_clean.pkl"
)

categories_clean.to_pickle(
    RUTA_INTERIM / "categories_clean.pkl"
)

holidays_clean.to_pickle(
    RUTA_INTERIM / "holidays_clean.pkl"
)

In [57]:
customers_clean["customer_zip_code_prefix"].head()

0    14409
1     9790
2     1151
3     8775
4    13056
Name: customer_zip_code_prefix, dtype: object

# Conclusiones del proceso de limpieza y transformación

## Objetivo

El objetivo de esta fase fue preparar los conjuntos de datos para las etapas posteriores de integración, creación de variables y análisis exploratorio, garantizando la calidad, consistencia e integridad de la información.

## Calidad general de los datos

La exploración y limpieza realizada mostró que los datos presentan un alto nivel de calidad. No se detectaron problemas graves de integridad, duplicidad o consistencia que comprometieran el análisis posterior.

Los principales problemas identificados fueron valores nulos en determinadas variables, algunos tipos de datos incorrectos y un pequeño número de registros con valores especiales que requirieron revisión.

## Transformación de tipos de datos

Se revisaron los tipos de datos de todas las tablas y se realizaron las conversiones necesarias.

Las columnas de fecha fueron convertidas a formato datetime para facilitar futuros análisis temporales y cálculos de diferencias entre fechas.

Los identificadores y códigos fueron tratados como variables de texto, ya que representan claves y no magnitudes numéricas.

Las variables físicas de producto fueron convertidas a formato entero tras completar el tratamiento de valores nulos.

## Tratamiento de valores nulos

Se eliminó la columna `isPaidTimeOff` del dataset de festivos al presentar un 100% de valores nulos.

En la tabla de productos se detectaron registros sin categoría y sin información descriptiva. Para preservar la información disponible se creó una categoría denominada `sin_categoria` y se imputaron los valores descriptivos faltantes.

Los valores nulos presentes en las fechas de aprobación y entrega de pedidos se mantuvieron, ya que se comprobó que están asociados a estados de pedido cancelados, no disponibles o en proceso, representando situaciones reales del negocio.

## Validación temporal

Se comprobó que los datos de pedidos cubren el periodo comprendido entre septiembre de 2016 y octubre de 2018.

El dataset de festivos fue filtrado para conservar únicamente los años presentes en los pedidos, garantizando la coherencia temporal entre ambas fuentes de información.

Además, se verificó que las principales variables temporales contienen información horaria relevante, por lo que se mantuvo la componente de hora en las fechas originales.

## Revisión de valores económicos

No se detectaron precios ni costes de envío negativos.

Se identificaron algunos registros con coste de envío igual a cero, representando únicamente el 0,34% del total de líneas de pedido. Estos registros se mantuvieron al considerarse compatibles con promociones o políticas de envío gratuito.

En la tabla de pagos se detectaron algunos registros con importe igual a cero asociados principalmente a cupones o métodos de pago no definidos. Dado su reducido volumen y su posible significado de negocio, se mantuvieron sin modificaciones.

También se identificaron dos registros con número de cuotas igual a cero asociados a pagos con tarjeta de crédito. Estos registros fueron corregidos asignando una única cuota por considerarse un error de captura.

## Duplicados e integridad referencial

No se detectaron duplicados completos en ninguno de los conjuntos de datos.

Asimismo, se verificó la integridad referencial entre todas las tablas principales, comprobando que no existen registros huérfanos en las relaciones entre clientes, pedidos, productos y pagos.

Esta validación garantiza que las futuras uniones entre tablas podrán realizarse sin pérdidas de información ni inconsistencias estructurales.

## Conclusión final

Tras el proceso de limpieza y transformación, los datasets presentan un elevado nivel de calidad, consistencia e integridad. Los datos quedan preparados para la siguiente fase del proyecto, centrada en la creación de variables analíticas, integración de tablas y construcción del dataset final que servirá de base para el análisis exploratorio, el estudio estadístico y el desarrollo del dashboard.
